# GradeScope Scraper Development Notebook

In [ ]:
from pathlib import Path
import os
import time
from typing import Optional

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from playwright.sync_api import sync_playwright, TimeoutError as PlaywrightTimeoutError
except ImportError:
    sync_playwright = None
    PlaywrightTimeoutError = TimeoutError

# Project Paths

In [ ]:
ROOT = Path.cwd()
SCRIPTS_DIR = ROOT / "scripts"
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CAPTURE_DIR = RAW_DIR / "portal_captures"
TEXT_DUMPS_DIR = RAW_DIR / "text_dumps"
SUMMARIES_DIR = DATA_DIR / "summaries"
PROCESSED_DIR = DATA_DIR / "processed"

for folder in [SCRIPTS_DIR, RAW_DIR, CAPTURE_DIR, TEXT_DUMPS_DIR, SUMMARIES_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

PORTAL_URL = "https://springzabdesk.szabist-isb.edu.pk/"
BASE_URL = "https://springzabdesk.szabist-isb.edu.pk"

print("Project root:", ROOT)
print("Capture folder:", CAPTURE_DIR)

# Safe Environment Template

In [ ]:
ENV_TEMPLATE = """
PORTAL_USERNAME=your_username_here
PORTAL_PASSWORD=your_password_here
""".strip()

ENV_EXAMPLE_PATH = ROOT / ".env.example"
ENV_EXAMPLE_PATH.write_text(ENV_TEMPLATE + "\n", encoding="utf-8")

print("Created safe placeholder file:", ENV_EXAMPLE_PATH)
print("Never commit a real .env file with portal credentials.")

# Optional Local Setup

In [ ]:
# Run these manually in a terminal only if Playwright is not installed:
# pip install playwright python-dotenv pandas beautifulsoup4
# python -m playwright install chromium

# Helper Functions

In [ ]:
def clean_filename(text: object) -> str:
    import re
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", str(text))
    return cleaned.strip("_").lower()


def absolute_url(href: Optional[str]) -> Optional[str]:
    if not href:
        return None
    if href.startswith("http"):
        return href
    if href.startswith("/"):
        return BASE_URL + href
    return BASE_URL + "/" + href


def extract_visible_text(page) -> str:
    try:
        return page.locator("body").inner_text(timeout=5000)
    except Exception:
        return ""


def save_capture(page, filename_prefix: str) -> dict:
    html_path = CAPTURE_DIR / f"{filename_prefix}.html"
    text_path = TEXT_DUMPS_DIR / f"{filename_prefix}.txt"

    html_path.write_text(page.content(), encoding="utf-8")
    text_path.write_text(extract_visible_text(page), encoding="utf-8")

    return {"html": html_path, "text": text_path}


def list_recent_captures(limit: int = 10) -> list[Path]:
    files = sorted(CAPTURE_DIR.glob("*.html"), key=lambda path: path.stat().st_mtime, reverse=True)
    return files[:limit]

# Login Detection Logic

In [ ]:
def wait_for_zabdesk_login(page, timeout_ms: int = 180000) -> None:
    page.wait_for_function(
        """
        () => {
            const links = Array.from(document.querySelectorAll("a"));
            const hasAttendance = links.some(a => a.innerText.includes("View Attendance"));
            const hasCurrentResults = links.some(a => a.innerText.includes("Current Semester Results"));
            const hasPreviousResults = links.some(a => a.innerText.includes("Previous Semesters Result"));
            return hasAttendance && hasCurrentResults && hasPreviousResults;
        }
        """,
        timeout=timeout_ms,
    )


def get_link_href(page, text_value: str) -> str:
    locator = page.locator("a", has_text=text_value).first
    href = locator.get_attribute("href")
    resolved = absolute_url(href)
    if not resolved:
        raise ValueError(f"Could not find link for {text_value}")
    return resolved

# Local Browser Smoke Test

In [ ]:
RUN_LOCAL_BROWSER_TEST = False

if RUN_LOCAL_BROWSER_TEST:
    if sync_playwright is None:
        raise RuntimeError("Playwright is not installed. Install it before running this cell.")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1440, "height": 940})
        page.goto(PORTAL_URL, wait_until="domcontentloaded")
        print("ZABDESK opened. Log in manually in the browser window.")
        wait_for_zabdesk_login(page)
        saved = save_capture(page, "notebook_logged_in_homepage")
        print("Saved:", saved)
        browser.close()
else:
    print("Local browser test skipped. Set RUN_LOCAL_BROWSER_TEST = True to run it locally.")

# Link Discovery Test

In [ ]:
RUN_LINK_DISCOVERY_TEST = False

if RUN_LINK_DISCOVERY_TEST:
    if sync_playwright is None:
        raise RuntimeError("Playwright is not installed. Install it before running this cell.")

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        page = browser.new_page(viewport={"width": 1440, "height": 940})
        page.goto(PORTAL_URL, wait_until="domcontentloaded")
        wait_for_zabdesk_login(page)

        attendance_url = get_link_href(page, "View Attendance")
        current_results_url = get_link_href(page, "Current Semester Results")
        previous_results_url = get_link_href(page, "Previous Semesters Result")

        print("Attendance URL:", attendance_url)
        print("Current results URL:", current_results_url)
        print("Previous results URL:", previous_results_url)
        browser.close()
else:
    print("Link discovery test skipped. Set RUN_LINK_DISCOVERY_TEST = True to run it locally.")

# Capture Inspection

In [ ]:
recent_captures = list_recent_captures(limit=10)

capture_index = [
    {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }
    for path in recent_captures
]

capture_index

# Text Dump Inspection

In [ ]:
text_dumps = sorted(TEXT_DUMPS_DIR.glob("*.txt"), key=lambda path: path.stat().st_mtime, reverse=True)

text_dump_index = [
    {
        "file": path.name,
        "size_kb": round(path.stat().st_size / 1024, 2),
        "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(path.stat().st_mtime)),
    }
    for path in text_dumps[:10]
]

text_dump_index

# Production Scraper Handoff

In [ ]:
production_files = [
    SCRIPTS_DIR / "portal_scraper.py",
    SCRIPTS_DIR / "parse_attendance.py",
    SCRIPTS_DIR / "parse_marks.py",
    SCRIPTS_DIR / "parse_gpa.py",
    SCRIPTS_DIR / "merge_dashboard.py",
    ROOT / "app.py",
]

status = [
    {
        "file": str(path.relative_to(ROOT)) if path.exists() else str(path),
        "exists": path.exists(),
    }
    for path in production_files
]

status

# Run Production Pipeline Locally

In [ ]:
# Run these commands in a terminal from the project root when you want the full local pipeline:
# python scripts/portal_scraper.py
# python scripts/parse_attendance.py
# python scripts/parse_marks.py
# python scripts/parse_gpa.py
# python scripts/merge_dashboard.py
# streamlit run app.py

# Final Notes

In [ ]:
notes = {
    "notebook_role": "Local scraper development and testing only",
    "production_scraper": "scripts/portal_scraper.py",
    "dashboard_app": "app.py",
    "privacy_rule": "Do not commit .env, raw portal captures, screenshots, or private academic CSV files",
}
notes